<a href="https://colab.research.google.com/github/mahendrapd/GRST/blob/main/BruteforceRST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
data = pd.read_parquet('/content/drive/MyDrive/dataset/training.parquet')

In [4]:
cols=len(data.columns)
nfeatures = cols-1
X = data.iloc[:,0:nfeatures]
y = data.iloc[:,-1]
print(len(data),cols)

347122 55


In [5]:
targetclass='Bruteforce'

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [7]:
def RSTClassifier(X,y,targetclass=y[0]):
    #print(targetclass)
    dc=pd.Series(y)
    cm=dc.groupby(dc).apply(lambda px: px.index.tolist()).to_dict() #index of targetclass
    D=set(cm[targetclass])
    # D1=set(cm['Bruteforce'])
    # D2=set(cm['Botnet'])
    # D3=set(cm['Portscan'])
    # D4=set(cm['Webattack'])
    # D5=set(cm['DoS'])
    # D6=set(cm['Infiltration'])
    # D7=set(cm['DDoS'])
    print(D)
    nfeatures=len(X.columns)
    #print(cm)
    purity={}
    headers = X.keys().tolist()

    for j in range(nfeatures):
        header=headers[j]
        hdata={}
        #print("Columns:",j,header)
        dp=pd.Series(X.iloc[:,j])
        md=dp.groupby(dp).apply(lambda px: px.index.tolist()).to_dict()
        K=md.keys()
        #print(K)
        for z in K:
            common_elements=set(md[z]) & D
            val=round(len(common_elements)/len(md[z]),3)
            #print(z,len(md[z]),len(common_elements),val)
            hdata[z]=val
        purity[header]=hdata
    #print(purity['urgent'][0])
    return purity

In [8]:
pur=RSTClassifier(X_train,y_train,targetclass)

{110595, 110597, 124967, 112136, 124425, 97803, 110604, 110605, 124440, 97306, 110623, 110624, 124961, 111139, 124964, 110629, 124966, 111140, 124968, 110633, 124970, 124969, 113195, 111661, 124973, 110634, 124976, 124972, 124971, 111147, 124975, 124974, 124990, 124991, 124992, 124481, 124993, 124995, 124480, 124997, 124994, 124999, 124998, 122953, 125002, 125003, 124996, 125007, 125008, 125010, 125011, 125012, 125013, 125014, 125015, 125019, 94320, 112761, 124555, 99986, 99989, 99990, 94358, 94366, 94374, 94391, 94922, 93908, 93909, 93910, 110805, 93913, 93914, 93915, 93916, 93918, 93920, 93961, 93962, 93963, 93979, 93983, 111910, 94010, 111420, 100173, 111452, 111972, 98678, 110979, 124301, 112013, 110992, 96165, 111021, 122302, 111555, 94159, 124889, 111067, 123361, 99299, 94190, 124963, 98293, 111096, 111097}


In [9]:
#print(pur)

In [10]:
def RST_AmbigiousSample(X,pur):
    headers = X.keys().tolist()
    print(headers,len(headers),len(X))
    ambigioussamples={}
    for i in range(len(X)):
        flag=0
        sumpurity=0
        for j in range(len(headers)):
            key=X.iloc[i][headers[j]]
            if (pur[headers[j]][key] == 1 or pur[headers[j]][key] == 0):
                flag=1
                break
            else:
                sumpurity+=pur[headers[j]][key]
        if (flag == 0):
            Avgpurity=round(sumpurity/len(headers),3)
            ambigioussamples[i]=Avgpurity
    return ambigioussamples

In [11]:
AmbSample=RST_AmbigiousSample(X_train,pur)

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle 

In [12]:
print(len(AmbSample))

0


In [13]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [14]:
def RST_Gradient(y,sample,epoch=300):
    Theta=0.5
    LRate=0.01
    pc=0
    for i in range(epoch):
        correctcount=0
        val=0
        Th=Theta
        L=0

        for key, value in sample.items():
            if (value >= Theta and y.iloc[key] == targetclass):
                correctcount+=1
            elif (value < Theta and y.iloc[key] != targetclass):
                correctcount+=1
            else:
                val+=value
        ictotal=(len(sample)-correctcount)
        try:
          p=correctcount/len(sample)
          #L=-(Th*math.log2(p)+(1-Th)*math.log2(1-p))
          #L=-((Th/p)-((1-Th)/(1-p)))
          #L=np.tanh(L)
          #Theta=abs(round((Theta-LRate*L),3))
          #print(L)
          if (pc<=correctcount):
            Theta=abs(round(Theta-LRate*(sigmoid((val/ictotal))),3))
            pc=correctcount
          #Theta=round(Theta-LRate*(0-(val/ictotal))*(val/ictotal)*(1-(val/ictotal)),3)
        except:
          Theta=Theta
        print(i, correctcount, ictotal, len(sample), val, Theta)

    return Theta

In [15]:
Th=RST_Gradient(y_train,AmbSample,300)

0 0 0 0 0 0.5
1 0 0 0 0 0.5
2 0 0 0 0 0.5
3 0 0 0 0 0.5
4 0 0 0 0 0.5
5 0 0 0 0 0.5
6 0 0 0 0 0.5
7 0 0 0 0 0.5
8 0 0 0 0 0.5
9 0 0 0 0 0.5
10 0 0 0 0 0.5
11 0 0 0 0 0.5
12 0 0 0 0 0.5
13 0 0 0 0 0.5
14 0 0 0 0 0.5
15 0 0 0 0 0.5
16 0 0 0 0 0.5
17 0 0 0 0 0.5
18 0 0 0 0 0.5
19 0 0 0 0 0.5
20 0 0 0 0 0.5
21 0 0 0 0 0.5
22 0 0 0 0 0.5
23 0 0 0 0 0.5
24 0 0 0 0 0.5
25 0 0 0 0 0.5
26 0 0 0 0 0.5
27 0 0 0 0 0.5
28 0 0 0 0 0.5
29 0 0 0 0 0.5
30 0 0 0 0 0.5
31 0 0 0 0 0.5
32 0 0 0 0 0.5
33 0 0 0 0 0.5
34 0 0 0 0 0.5
35 0 0 0 0 0.5
36 0 0 0 0 0.5
37 0 0 0 0 0.5
38 0 0 0 0 0.5
39 0 0 0 0 0.5
40 0 0 0 0 0.5
41 0 0 0 0 0.5
42 0 0 0 0 0.5
43 0 0 0 0 0.5
44 0 0 0 0 0.5
45 0 0 0 0 0.5
46 0 0 0 0 0.5
47 0 0 0 0 0.5
48 0 0 0 0 0.5
49 0 0 0 0 0.5
50 0 0 0 0 0.5
51 0 0 0 0 0.5
52 0 0 0 0 0.5
53 0 0 0 0 0.5
54 0 0 0 0 0.5
55 0 0 0 0 0.5
56 0 0 0 0 0.5
57 0 0 0 0 0.5
58 0 0 0 0 0.5
59 0 0 0 0 0.5
60 0 0 0 0 0.5
61 0 0 0 0 0.5
62 0 0 0 0 0.5
63 0 0 0 0 0.5
64 0 0 0 0 0.5
65 0 0 0 0 0.5
66 0 0 0 0 0.5
67 0 

In [16]:
def RST_Validation(X,y,pur,Theta):
    TP=0
    TN=0
    FP=0
    FN=0
    headers = X.keys().tolist()
    for i in range(len(X)):
        val=0
        count=0
        pval=0

        flag=0
        for j in range(len(headers)):
            #KYS=list(pur[headers[j]].keys())
            #dflag=0
            key=X.iloc[i][headers[j]]
            try:
              pval=pur[headers[j]][key]
            except KeyError:
              pval=Theta

            if ((pval == 1 or pval == 0)):
                flag=1
                break
            else:
                val+=pval
                count+=1

        if (flag == 1):
            if (pval==1 and y.iloc[i] == targetclass):
                TP+=1
            elif (pval==1 and y.iloc[i] != targetclass):
                FN+=1
            elif (pval==0 and y.iloc[i] != targetclass):
                TN+=1
            else:
                FP+=1
        else:
            Aval=round((val/count),3)
            if (Aval>=Theta and y.iloc[i] == targetclass):
                TP+=1
            elif (Aval<Theta and y.iloc[i] != targetclass):
                TN+=1
            elif (Aval>=Theta and y.iloc[i] != targetclass):
                FN+=1
            else:
                FP+=1
    try:
      Recall=round(TP/(TP+FN),4)
    except:
      Recall=0
    try:
      FPR=round(FP/(TN+FP),4)
    except:
      FPR=0
    try:
      Precision=round(TP/(TP+FP),4)
    except:
      Precision=0
    try:
      Accuracy=round((TP+TN)/(TP+TN+FP+FN),4)
    except:
      Accuracy=0
    try:
      FScore=round(2*Recall*Precision/(Recall+Precision),4)
    except:
      FScore=0
    print("TP=",TP,"\tTN=",TN,"\tFP=",FP,"\tFN=",FN)
    return Recall, FPR, Precision, FScore, Accuracy

In [17]:
Result=RST_Validation(X_train,y_train,pur,Th)
print(Result)

TP= 0 	TN= 260234 	FP= 107 	FN= 0
(0, 0.0004, 0.0, 0, 0.9996)


In [18]:
ResultTest = RST_Validation(X_test,y_test,pur,Th)
print(ResultTest)

TP= 0 	TN= 86749 	FP= 32 	FN= 0
(0, 0.0004, 0.0, 0, 0.9996)


In [19]:
def CountPatterns(pur):
  value=0
  for outer_key, outer_value in pur.items():
    acount=0
    ncount=0
    for inner_key, inner_value in outer_value.items():
      if (inner_value == 0):
        acount=acount+1
      if (inner_value == 1):
        ncount=ncount+1
    print(f"{outer_key}: {acount}, {ncount}, {len(outer_value.items())}")

In [20]:
CountPatterns(pur)

Flow Duration: 11, 0, 11
Total Fwd Packets: 75, 0, 75
Total Backward Packets: 47, 0, 47
Fwd Packets Length Total: 11, 0, 11
Bwd Packets Length Total: 38, 0, 38
Fwd Packet Length Max: 41, 0, 42
Fwd Packet Length Mean: 44, 0, 45
Fwd Packet Length Std: 41, 0, 42
Bwd Packet Length Max: 27, 0, 29
Bwd Packet Length Mean: 16, 0, 17
Bwd Packet Length Std: 29, 0, 31
Flow Bytes/s: 91, 0, 92
Flow Packets/s: 24, 0, 39
Flow IAT Mean: 5, 0, 5
Flow IAT Std: 11, 0, 11
Flow IAT Max: 11, 0, 11
Flow IAT Min: 9, 0, 9
Fwd IAT Total: 11, 0, 11
Fwd IAT Mean: 5, 0, 5
Fwd IAT Std: 11, 0, 11
Fwd IAT Max: 11, 0, 11
Fwd IAT Min: 9, 0, 9
Bwd IAT Total: 91, 0, 101
Bwd IAT Mean: 100, 0, 101
Bwd IAT Std: 98, 0, 101
Bwd IAT Max: 97, 0, 101
Bwd IAT Min: 101, 0, 101
Fwd Header Length: 89, 0, 89
Bwd Header Length: 9, 0, 9
Fwd Packets/s: 32, 0, 46
Bwd Packets/s: 23, 0, 35
Packet Length Max: 45, 0, 47
Packet Length Mean: 21, 0, 22
Packet Length Std: 29, 0, 31
Packet Length Variance: 11, 0, 11
Avg Packet Size: 22, 0, 23
Avg